# 04 — OSM Dataset Preparation

## 1. Purpose

This notebook prepares the final OpenStreetMap dataset used by ColMaps.

The previous stages inspected the raw Colombia OSM dataset, defined the
application-level feature scope, and validated potentially ambiguous candidate
mappings. This stage converts those semantic decisions into a deterministic
data-preparation pipeline applied to the original OSM source dataset.

The preparation process uses the validated feature-scope specification to:

* retain mappings accepted during feature-scope validation;
* remove mappings explicitly excluded from the ColMaps scope;
* apply validated refinement rules where additional filtering is required;
* preserve secondary classifications required by refined natural features;
* maintain the OSM structural information required for subsequent geospatial
  processing.

The output of this stage is the final prepared OSM extract representing the
validated ColMaps feature population. This dataset is intended for subsequent
processing and import into PostgreSQL/PostGIS.

### Inputs

* `raw/colombia-latest.osm.pbf`
* `filters/02_validated_feature_scope.csv`

### Output

* `processed/colombia-colmaps.osm.pbf`

This notebook performs deterministic dataset preparation. It does not redefine
the semantic feature scope established during the previous validation stages.

## 2. Setup and Input Validation

The preparation pipeline operates on the original Colombia `.osm.pbf` dataset
rather than the intermediate candidate extract. This ensures that the final
dataset is generated directly from the original source using the validated
feature-scope specification.

Before constructing the filtering operations, the required input files and the
validated scope are checked for consistency.



In [1]:
from pathlib import Path

import pandas as pd


RAW_PBF_PATH = Path("../raw/colombia-260901.osm.pbf")
VALIDATED_SCOPE_PATH = Path("../filters/02_validated_feature_scope.csv")

OUTPUT_DIR = Path("../processed")
OUTPUT_PBF_PATH = OUTPUT_DIR / "colombia-colmaps.osm.pbf"


# Verify that the required pipeline inputs exist.
if not RAW_PBF_PATH.exists():
    raise FileNotFoundError(
        f"Raw OSM dataset not found: {RAW_PBF_PATH}"
    )

if not VALIDATED_SCOPE_PATH.exists():
    raise FileNotFoundError(
        f"Validated feature scope not found: {VALIDATED_SCOPE_PATH}"
    )

# Ensure the output directory exists.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Raw dataset:     {RAW_PBF_PATH}")
print(f"Validated scope: {VALIDATED_SCOPE_PATH}")
print(f"Output dataset:  {OUTPUT_PBF_PATH}")

Raw dataset:     ..\raw\colombia-260901.osm.pbf
Validated scope: ..\filters\02_validated_feature_scope.csv
Output dataset:  ..\processed\colombia-colmaps.osm.pbf


In [2]:
validated_scope = pd.read_csv(VALIDATED_SCOPE_PATH)

print(f"Validated mappings: {len(validated_scope):,}")

display(validated_scope.head())

Validated mappings: 115


,category,osm_key,osm_value,include,reason,decision,secondary_key,rule_type,excluded_values
0,attractions,tourism,attraction,True,Explicit OSM classification for tourist attrac...,RETAIN,NaN,NaN,NaN
1,attractions,man_made,lighthouse,True,Lighthouses represent distinctive landmarks wi...,RETAIN,NaN,NaN,NaN
2,attractions,man_made,observatory,True,Observatories represent distinctive visitor-or...,RETAIN,NaN,NaN,NaN
3,viewpoints,tourism,viewpoint,True,Explicit OSM classification for scenic viewpoi...,RETAIN,NaN,NaN,NaN
4,museums_culture,tourism,museum,True,Museums represent explicit visitor-oriented cu...,RETAIN,NaN,NaN,NaN


In [3]:
## 1. Let's check the validation is valid
required_columns = {
    "category",
    "osm_key",
    "osm_value",
    "decision",
    "secondary_key",
    "rule_type",
    "excluded_values",
}

missing_columns = required_columns - set(validated_scope.columns)

if missing_columns:
    raise ValueError(
        f"Validated scope is missing required columns: "
        f"{sorted(missing_columns)}"
    )


# Validate the decision vocabulary produced by notebook 03.
valid_decisions = {"RETAIN", "REFINE", "EXCLUDE"}

unexpected_decisions = (
    set(validated_scope["decision"].dropna())
    - valid_decisions
)

if unexpected_decisions:
    raise ValueError(
        f"Unexpected validation decisions: "
        f"{sorted(unexpected_decisions)}"
    )


decision_summary = (
    validated_scope["decision"]
    .value_counts()
    .reindex(["RETAIN", "REFINE", "EXCLUDE"], fill_value=0)
    .rename_axis("decision")
    .reset_index(name="mapping_count")
)

display(decision_summary)

,decision,mapping_count
0,RETAIN,110
1,REFINE,4
2,EXCLUDE,1


## 3. Primary Feature Extraction

The validated feature-scope specification is used to construct the primary OSM
filter expressions.

Mappings classified as `RETAIN` or `REFINE` remain part of the ColMaps feature
population and are therefore included in the initial extraction. Mappings
classified as `EXCLUDE` are omitted.

Refinement conditions are not applied during this initial extraction. Refined
mappings must first remain available in the prepared population so that their
validated secondary attributes can be evaluated during the subsequent
refinement stage.

The extraction preserves referenced OSM elements required by matching ways and
relations, allowing their geometries and structural relationships to remain
reconstructable during subsequent processing.


In [4]:
# Select mappings that remain part of the validated ColMaps scope.
active_scope = validated_scope[
    validated_scope["decision"].isin(["RETAIN", "REFINE"])
].copy()

print(f"Active mappings: {len(active_scope):,}")
print(
    f"Excluded mappings: "
    f"{(validated_scope['decision'] == 'EXCLUDE').sum():,}"
)

Active mappings: 114
Excluded mappings: 1


In [5]:
# Convert the validated mappings into Osmium tag-filter expressions.
filter_expressions = (
    active_scope["osm_key"]
    + "="
    + active_scope["osm_value"]
).tolist()

print(f"Filter expressions generated: {len(filter_expressions):,}")

filter_expressions[:10]

Filter expressions generated: 114


['tourism=attraction',
 'man_made=lighthouse',
 'man_made=observatory',
 'tourism=viewpoint',
 'tourism=museum',
 'tourism=gallery',
 'tourism=artwork',
 'amenity=theatre',
 'historic=monument',
 'historic=memorial']

In [6]:
## Sanity Check 
# Ensure that excluded mappings cannot accidentally enter the primary filter.
excluded_expressions = set(
    validated_scope.loc[
        validated_scope["decision"] == "EXCLUDE",
        "osm_key",
    ]
    + "="
    + validated_scope.loc[
        validated_scope["decision"] == "EXCLUDE",
        "osm_value",
    ]
)

assert not excluded_expressions.intersection(filter_expressions)

print("Primary filter validation passed.")
print(f"Excluded expressions: {sorted(excluded_expressions)}")

Primary filter validation passed.
Excluded expressions: ['amenity=parking_entrance']


### 3.1 Validated Primary Scope Extraction

The active mapping expressions are applied to the original Colombia OSM dataset
to produce an intermediate extract containing the validated primary feature
population.

Referenced nodes and relation members are preserved during this extraction so
that matching ways and relations remain structurally complete. Consequently,
this intermediate file may contain supporting OSM elements that do not
independently match a ColMaps feature mapping.

The resulting extract is an intermediate preparation artifact. Refinement rules
for mappings classified as `REFINE` are applied in the subsequent stage before
the final ColMaps dataset is produced.


In [7]:
import subprocess


PRIMARY_SCOPE_PATH = OUTPUT_DIR / "01_primary_scope.osm.pbf"

result = subprocess.run(
    [
        "osmium",
        "tags-filter",
        str(RAW_PBF_PATH),
        *filter_expressions,
        "-o",
        str(PRIMARY_SCOPE_PATH),
        "--overwrite",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

print(f"Primary scope extracted to: {PRIMARY_SCOPE_PATH}")

Primary scope extracted to: ..\processed\01_primary_scope.osm.pbf


In [8]:
if not PRIMARY_SCOPE_PATH.exists():
    raise FileNotFoundError(
        f"Primary scope extract was not created: {PRIMARY_SCOPE_PATH}"
    )

primary_size_mb = PRIMARY_SCOPE_PATH.stat().st_size / (1024 ** 2)
raw_size_mb = RAW_PBF_PATH.stat().st_size / (1024 ** 2)

reduction_percent = (
    1 - PRIMARY_SCOPE_PATH.stat().st_size / RAW_PBF_PATH.stat().st_size
) * 100

print(f"Raw dataset:   {raw_size_mb:.2f} MiB")
print(f"Primary scope: {primary_size_mb:.2f} MiB")
print(f"Size reduction: {reduction_percent:.1f}%")

Raw dataset:   313.28 MiB
Primary scope: 25.81 MiB
Size reduction: 91.8%


## 4. Refinement Rules Application

The primary extract contains all mappings retained by the validated ColMaps
feature scope. Mappings classified as `REFINE` now require the additional
conditions established during targeted validation.

Two refinement types are represented in the validated specification:

* **classification** — the feature remains included, while its validated
  secondary classification must be preserved for subsequent semantic
  processing;
* **restriction** — explicit secondary values identify features that should not
  remain in the generally accessible ColMaps destination population.

Classification refinements therefore do not remove features from the primary
extract. Restriction refinements may remove objects when the validated
exclusion condition is explicitly present.

Missing secondary attributes are not interpreted as exclusion conditions.


In [9]:
# Select the refinement rules produced by notebook 03.
refined_scope = validated_scope[
    validated_scope["decision"] == "REFINE"
].copy()

refinement_summary = refined_scope[
    [
        "category",
        "osm_key",
        "osm_value",
        "secondary_key",
        "rule_type",
        "excluded_values",
    ]
]

display(refinement_summary)

,category,osm_key,osm_value,secondary_key,rule_type,excluded_values
26,nature,natural,water,water,classification,NaN
27,nature,natural,wetland,wetland,classification,NaN
60,parks_recreation,leisure,garden,access,restriction,private|no
65,parks_recreation,leisure,swimming_pool,access,restriction,private|no


In [10]:
## Split the 4 REFINE process
classification_rules = refined_scope[
    refined_scope["rule_type"] == "classification"
].copy()

restriction_rules = refined_scope[
    refined_scope["rule_type"] == "restriction"
].copy()

print(
    f"Classification refinements: {len(classification_rules)}"
)
print(
    f"Restriction refinements:    {len(restriction_rules)}"
)

display(classification_rules)
display(restriction_rules)

Classification refinements: 2
Restriction refinements:    2


,category,osm_key,osm_value,include,reason,decision,secondary_key,rule_type,excluded_values
26,nature,natural,water,True,Water bodies may represent scenic natural dest...,REFINE,water,classification,NaN
27,nature,natural,wetland,True,Wetlands may represent natural areas of ecolog...,REFINE,wetland,classification,NaN


,category,osm_key,osm_value,include,reason,decision,secondary_key,rule_type,excluded_values
60,parks_recreation,leisure,garden,True,Gardens represent landscaped recreational spac...,REFINE,access,restriction,private|no
65,parks_recreation,leisure,swimming_pool,True,Swimming pools represent recreational faciliti...,REFINE,access,restriction,private|no


### 4.1 Restriction Refinements

Restriction refinements identify features that remain valid ColMaps mappings in
general but contain explicit secondary attributes indicating restricted
accessibility.

The validated specification currently applies this rule to
`leisure=swimming_pool` and `leisure=garden`. Objects explicitly tagged with
`access=private` or `access=no` are excluded from the generally accessible
ColMaps feature population.

The restriction rules are derived directly from the validated feature-scope
specification rather than being independently redefined in this preparation
stage.


In [11]:
# Expand the restriction rules into explicit primary/secondary
# combinations that must be excluded.
restriction_expressions = []

for _, rule in restriction_rules.iterrows():
    excluded_values = str(rule["excluded_values"]).split("|")

    for excluded_value in excluded_values:
        restriction_expressions.append({
            "osm_key": rule["osm_key"],
            "osm_value": rule["osm_value"],
            "secondary_key": rule["secondary_key"],
            "secondary_value": excluded_value,
        })

restriction_expressions_df = pd.DataFrame(
    restriction_expressions
)

display(restriction_expressions_df)

,osm_key,osm_value,secondary_key,secondary_value
0,leisure,garden,access,private
1,leisure,garden,access,no
2,leisure,swimming_pool,access,private
3,leisure,swimming_pool,access,no


In [12]:
from pyrosm import OSM


# Read only the mappings that require access-based refinement.
osm = OSM(
    str(PRIMARY_SCOPE_PATH),
    engine="out_of_core",
    workers=1,
)

restricted_candidates = osm.get_data_by_custom_criteria(
    custom_filter={
        "leisure": [
            "swimming_pool",
            "garden",
        ],
    },
    tags_as_columns=[
        "access",
        "leisure",
        "name",
    ],
    keep_nodes=True,
    keep_ways=True,
    keep_relations=True,
    keep_other_tags=False,
)

print(
    f"Restriction candidates loaded: "
    f"{len(restricted_candidates):,}"
)

Restriction candidates loaded: 14,348


In [13]:
restricted_access_values = set()

for values in restriction_rules["excluded_values"].dropna():
    restricted_access_values.update(
        str(values).split("|")
    )

print(
    f"Restricted access values: "
    f"{sorted(restricted_access_values)}"
)


restricted_features = restricted_candidates[
    restricted_candidates["access"].isin(
        restricted_access_values
    )
].copy()

restriction_counts = (
    restricted_features
    .groupby(["leisure", "access"])
    .size()
    .reset_index(name="feature_count")
)

display(restriction_counts)

print(
    f"Total features marked for exclusion: "
    f"{len(restricted_features):,}"
)

Restricted access values: ['no', 'private']


,leisure,access,feature_count
0,garden,no,3
1,garden,private,38
2,swimming_pool,no,8
3,swimming_pool,private,900


Total features marked for exclusion: 949


### 4.2 Restricted Feature Removal

The access-based refinement identified 946 features explicitly classified as
`access=private` or `access=no`. These objects must not remain as independent
ColMaps destinations in the final prepared feature population.

Removing OSM objects requires additional care because nodes may participate in
ways and relation members may reference other OSM elements. The preparation
stage must therefore distinguish between removing a feature classification and
removing the underlying OSM object required for structural completeness.

To preserve valid OSM structure, restricted features are not blindly deleted
from the intermediate extract. Instead, the final feature population is
constructed from the validated inclusion rules while ensuring that restricted
objects are not selected as ColMaps features and that supporting referenced
elements required by retained ways and relations remain available.


In [14]:
# Inspect the OSM object types affected by the restriction rules.
restricted_type_counts = (
    restricted_features["osm_type"]
    .value_counts()
    .rename_axis("osm_type")
    .reset_index(name="feature_count")
)

display(restricted_type_counts)

,osm_type,feature_count
0,way,939
1,node,9
2,relation,1


In [15]:
# Verify that every restricted feature has an OSM identifier and type.
assert restricted_features["id"].notna().all()
assert restricted_features["osm_type"].notna().all()

restricted_features[
    [
        "id",
        "osm_type",
        "leisure",
        "access",
        "name",
    ]
].head(10)

,id,osm_type,leisure,access,name
14,1743423407,node,swimming_pool,private,NaN
21,2268356956,node,swimming_pool,private,Natural River Pool
37,3816216815,node,swimming_pool,private,NaN
200,9563065039,node,garden,private,Pot de fleur
201,9563076328,node,garden,private,Pot de fleurs
202,9563085057,node,garden,private,Pot de fleurs
285,12842605255,node,swimming_pool,private,NaN
286,12842609213,node,swimming_pool,private,NaN
293,13078308765,node,swimming_pool,private,ESTADERO PIEDRA LIZA
305,24546994,way,swimming_pool,private,Jorge E Cavelier


#### Restriction refinement result

The validated access restrictions identified 949 features that must not be
treated as generally accessible ColMaps destinations. The affected population
consists of 939 ways, 9 nodes, and 1 relation.

Because OSM objects may participate in structural relationships independently
of their ColMaps feature eligibility, these objects are not blindly removed
from the intermediate PBF by identifier. Their exclusion is instead preserved
as part of the validated feature-selection rules applied when constructing the
ColMaps feature population for database import.

This distinction preserves OSM structural integrity while ensuring that
features explicitly tagged with `access=private` or `access=no` are not exposed
as generally accessible traveler destinations.

### 4.3 Classification Refinements

The classification refinements for `natural=water` and `natural=wetland`
require their secondary `water=*` and `wetland=*` classifications to be
considered during final feature preparation.

Unlike access-based restrictions, these refinements do not identify individual
features as explicitly inaccessible. Instead, the secondary classifications
distinguish heterogeneous geographic environments represented by the same
primary OSM mapping.

The preparation stage therefore evaluates these secondary classifications to
determine which feature types should remain eligible for the ColMaps `Nature`
category while preventing unsuitable or unrelated classifications from being
treated as traveler-oriented natural features.


In [16]:
# Verify the classification attributes required by the validated scope.
expected_classification_keys = {"water", "wetland"}

classification_keys = set(
    classification_rules["secondary_key"].dropna()
)

assert classification_keys == expected_classification_keys

print(
    "Classification refinements preserved:",
    sorted(classification_keys),
)

Classification refinements preserved: ['water', 'wetland']


#### 4.3.1 `natural=water`

The `natural=water` mapping represents a heterogeneous population of water
features. Targeted validation identified `water=*` as the secondary
classification required to distinguish different water-body types.

For dataset preparation, secondary classifications are evaluated according to
their suitability as geographic features within the ColMaps `Nature` category.
The objective is to retain recognizable water bodies that may provide
route-oriented discovery value while preventing infrastructure-oriented,
artificial, or otherwise unsuitable classifications from being represented as
independent natural destinations.

Objects without a usable `water=*` classification are handled separately
because the absence of the secondary tag does not itself establish that the
feature is unsuitable.


In [17]:
# Inspect the secondary water classifications present in the
# prepared primary feature population.
result = subprocess.run(
    [
        "osmium",
        "tags-count",
        "--sort=count-desc",
        str(PRIMARY_SCOPE_PATH),
        "water=*",
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

water_rows = []

for line in result.stdout.strip().splitlines():
    parts = line.split("\t")

    if len(parts) != 3:
        continue

    count, key, value = parts

    water_rows.append({
        "value": value.strip('"'),
        "count": int(count),
    })

water_values = pd.DataFrame(water_rows)

display(water_values)

,value,count
0,pond,12063
1,river,3786
2,lake,2788
3,reservoir,2017
4,basin,221
5,oxbow,205
6,canal,118
7,wastewater,108
8,fishpond,74
9,lagoon,57


The observed `water=*` values contain both standardized water-body
classifications and non-standard free-text values. The preparation rule
therefore uses an explicit set of accepted secondary classifications rather
than treating every observed value as semantically equivalent.

The following classifications are retained as recognizable water features:
`pond`, `river`, `lake`, `reservoir`, `oxbow`, `lagoon`, `stream`, and
`stream_pool`.

Infrastructure-oriented or specialized classifications such as `basin`,
`canal`, `wastewater`, `fishpond`, `pool`, `reflecting_pool`, `swale`, `drain`,
and `moat` are not treated as independent ColMaps `Nature` features through
this mapping. Non-standard or free-text `water=*` values are likewise not
automatically accepted as semantic water classifications.

Objects where `water=*` is absent are not automatically excluded. Missing
secondary information is treated as an unclassified `natural=water` feature
rather than evidence that the object is unsuitable.

This produces the following preparation rule:

* recognized and accepted `water=*` → retain;
* recognized but unsuitable `water=*` → exclude from the ColMaps feature
  population;
* non-standard or ambiguous `water=*` → exclude from automatic feature
  classification;
* missing `water=*` → retain as unclassified water.


In [18]:
# Valid secondary classifications for natural=water.
allowed_water_values = {
    "pond",
    "river",
    "lake",
    "reservoir",
    "oxbow",
    "lagoon",
    "stream",
    "stream_pool",
}

# Values with an explicit water=* classification are accepted only
# when that classification belongs to the validated allowlist.
# Missing water=* remains eligible as unclassified natural water.

In [19]:
# Load natural=water features from the prepared primary scope.
water_osm = OSM(
    str(PRIMARY_SCOPE_PATH),
    engine="out_of_core",
    workers=1,
)

water_features = water_osm.get_data_by_custom_criteria(
    custom_filter={
        "natural": ["water"],
    },
    tags_as_columns=[
        "natural",
        "water",
        "name",
    ],
    keep_nodes=True,
    keep_ways=True,
    keep_relations=True,
    keep_other_tags=False,
)

print(f"natural=water features loaded: {len(water_features):,}")

natural=water features loaded: 42,520


In [20]:
# Classify each natural=water feature according to the
# validated secondary-classification rule.
water_features["preparation_status"] = "EXCLUDE"

# Missing water=* is retained as unclassified water.
missing_water_mask = water_features["water"].isna()

water_features.loc[
    missing_water_mask,
    "preparation_status"
] = "RETAIN_UNCLASSIFIED"

# Explicitly accepted water=* classifications are retained.
allowed_water_mask = water_features["water"].isin(
    allowed_water_values
)

water_features.loc[
    allowed_water_mask,
    "preparation_status"
] = "RETAIN_CLASSIFIED"

In [21]:
excluded_water_values = (
    water_features.loc[
        water_features["preparation_status"] == "EXCLUDE",
        "water",
    ]
    .value_counts(dropna=False)
    .rename_axis("water")
    .reset_index(name="feature_count")
)

display(excluded_water_values)

,water,feature_count
0,basin,221
1,canal,118
2,wastewater,108
3,fishpond,74
4,pool,12
5,wetland,7
6,reflecting_pool,7
7,swale,6
8,Quebrada Piaché,3
9,moat,3


In [22]:
# Reclassify water features that explicitly represent wetlands.
wetland_reclassification_mask = (
    water_features["water"] == "wetland"
)

water_features.loc[
    wetland_reclassification_mask,
    "preparation_status"
] = "RECLASSIFY_WETLAND"

In [23]:
water_preparation_summary = (
    water_features["preparation_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="feature_count")
)

display(water_preparation_summary)

,status,feature_count
0,RETAIN_UNCLASSIFIED,21195
1,RETAIN_CLASSIFIED,20732
2,EXCLUDE,586
3,RECLASSIFY_WETLAND,7


The final `natural=water` preparation rule retains recognized water-body
classifications and unclassified water features while preventing explicitly
unsuitable or non-standard secondary classifications from being automatically
treated as ColMaps natural destinations.

Recognized classifications retained as water features are `pond`, `river`,
`lake`, `reservoir`, `oxbow`, `lagoon`, `stream`, and `stream_pool`.

Objects tagged with `water=wetland` are not discarded. They are marked for
semantic reclassification so that they can be represented consistently with
the wetland feature population during the subsequent ColMaps data
transformation.

Explicit `water=*` values outside the accepted classification set are excluded
from automatic water-feature eligibility. Objects without a `water=*`
classification remain eligible as unclassified water because missing secondary
information does not provide evidence of semantic unsuitability.

Applied to the prepared dataset, the rule produces:

* 20,732 classified water features retained;
* 21,195 unclassified water features retained;
* 7 features marked for wetland reclassification;
* 586 features excluded from water-feature eligibility.
status	feature_count

#### 4.3.2 `natural=wetland`

The `natural=wetland` mapping represents different types of wetland
environments. Targeted validation identified `wetland=*` as the relevant
secondary classification for distinguishing these environments.

Unlike `natural=water`, the dominant secondary values observed for this mapping
largely describe recognizable wetland types. The preparation stage evaluates
these classifications to preserve valid wetland environments while preventing
non-standard or semantically unrelated values from being automatically treated
as ColMaps natural features.

Objects without a `wetland=*` classification are evaluated separately because
missing secondary information does not itself indicate that a
`natural=wetland` feature is unsuitable.

Features represented through the alternative
`natural=water + water=wetland` convention are subsequently normalized to the
same ColMaps wetland classification.



In [24]:
# Load natural=wetland features from the prepared primary scope.
wetland_features = water_osm.get_data_by_custom_criteria(
    custom_filter={
        "natural": ["wetland"],
    },
    tags_as_columns=[
        "natural",
        "wetland",
        "name",
    ],
    keep_nodes=True,
    keep_ways=True,
    keep_relations=True,
    keep_other_tags=False,
)

print(
    f"natural=wetland features loaded: "
    f"{len(wetland_features):,}"
)

natural=wetland features loaded: 8,203


In [25]:
wetland_value_counts = (
    wetland_features["wetland"]
    .value_counts(dropna=False)
    .rename_axis("wetland")
    .reset_index(name="feature_count")
)

display(wetland_value_counts)

,wetland,feature_count
0,NaN,4086
1,mangrove,2101
2,wet_meadow,533
3,tidalflat,441
4,reedbed,383
5,swamp,324
6,marsh,140
7,bog,111
8,saltmarsh,20
9,saltern,16


In [26]:
# Valid secondary classifications for natural=wetland.
allowed_wetland_values = {
    "mangrove",
    "wet_meadow",
    "tidalflat",
    "reedbed",
    "swamp",
    "marsh",
    "bog",
    "saltmarsh",
    "fen",
}


# Classify each natural=wetland feature according to the
# validated secondary-classification rule.
wetland_features["preparation_status"] = "EXCLUDE"

# Missing wetland=* remains eligible as an unclassified wetland.
missing_wetland_mask = wetland_features["wetland"].isna()

wetland_features.loc[
    missing_wetland_mask,
    "preparation_status"
] = "RETAIN_UNCLASSIFIED"

# Recognized wetland classifications remain eligible.
allowed_wetland_mask = wetland_features["wetland"].isin(
    allowed_wetland_values
)

wetland_features.loc[
    allowed_wetland_mask,
    "preparation_status"
] = "RETAIN_CLASSIFIED"

In [27]:
wetland_preparation_summary = (
    wetland_features["preparation_status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="feature_count")
)

display(wetland_preparation_summary)

print(f"Total natural=wetland: {len(wetland_features):,}")

print(
    "Retained:",
    (
        wetland_features["preparation_status"]
        != "EXCLUDE"
    ).sum(),
)

print(
    "Excluded:",
    (
        wetland_features["preparation_status"]
        == "EXCLUDE"
    ).sum(),
)

,status,feature_count
0,RETAIN_UNCLASSIFIED,4086
1,RETAIN_CLASSIFIED,4056
2,EXCLUDE,61


Total natural=wetland: 8,203
Retained: 8142
Excluded: 61


In [28]:
## Check what we reject
excluded_wetland_values = (
    wetland_features.loc[
        wetland_features["preparation_status"] == "EXCLUDE",
        "wetland",
    ]
    .value_counts(dropna=False)
    .rename_axis("wetland")
    .reset_index(name="feature_count")
)

display(excluded_wetland_values)

,wetland,feature_count
0,saltern,16
1,lake,4
2,arroyo,4
3,waterfall,3
4,P_4,3
5,pond,3
6,humedal,2
7,espejo_de_agua,2
8,mud,2
9,yes,2


#### Preparation result

The validated wetland classification rule retains the recognized secondary
classifications `mangrove`, `wet_meadow`, `tidalflat`, `reedbed`, `swamp`,
`marsh`, `bog`, `saltmarsh`, and `fen`.

Features without a `wetland=*` classification remain eligible as unclassified
wetlands because missing secondary information is not interpreted as evidence
of semantic unsuitability. Explicit secondary values outside the accepted
classification set are excluded from automatic wetland-feature eligibility.

Applied to the prepared population, the rule produces:

* 4,086 classified wetland features retained;
* 4,086 unclassified wetland features retained;
* 61 features excluded from wetland-feature eligibility.

Additionally, the 7 features identified through
`natural=water + water=wetland` are preserved for semantic reclassification
into the same ColMaps wetland feature type during subsequent transformation.

### 4.4 Preparation Rule Specification

The refinement decisions applied during dataset preparation are exported as a
machine-readable rule specification.

The validated feature-scope CSV determines which primary OSM mappings belong to
ColMaps, while the preparation-rule specification defines the additional
conditions required by mappings classified as `REFINE`.

Keeping these responsibilities separate avoids embedding implementation rules
directly into the feature-scope definition and allows the subsequent data
transformation and PostGIS import stages to consume the same validated
preparation logic.

The resulting specification is stored as
`filters/03_preparation_rules.json`.



In [29]:
import json


PREPARATION_RULES_PATH = Path(
    "../filters/03_preparation_rules.json"
)

preparation_rules = {
    "natural=water": {
        "rule_type": "classification",
        "secondary_key": "water",
        "allowed_values": sorted(allowed_water_values),
        "missing_value": "retain",
        "reclassify": {
            "wetland": "wetland",
        },
    },

    "natural=wetland": {
        "rule_type": "classification",
        "secondary_key": "wetland",
        "allowed_values": sorted(allowed_wetland_values),
        "missing_value": "retain",
    },

    "leisure=swimming_pool": {
        "rule_type": "restriction",
        "secondary_key": "access",
        "excluded_values": ["private", "no"],
        "missing_value": "retain",
    },

    "leisure=garden": {
        "rule_type": "restriction",
        "secondary_key": "access",
        "excluded_values": ["private", "no"],
        "missing_value": "retain",
    },
}

In [30]:
with PREPARATION_RULES_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preparation_rules,
        file,
        indent=2,
        ensure_ascii=False,
    )

print(
    f"Preparation rules exported to: "
    f"{PREPARATION_RULES_PATH}"
)

Preparation rules exported to: ..\filters\03_preparation_rules.json


In [31]:
## Sanity Check to see if everything is okay
refined_mappings = set(
    refined_scope["osm_key"]
    + "="
    + refined_scope["osm_value"]
)

specified_mappings = set(
    preparation_rules.keys()
)

assert refined_mappings == specified_mappings, (
    "Preparation rules do not match the validated "
    "REFINE mappings."
)

print("All REFINE mappings have preparation rules.")

All REFINE mappings have preparation rules.


## 5. Final Dataset Validation

The final preparation stage verifies that the generated dataset and semantic
rule specifications are consistent with the validated ColMaps feature scope.

These checks ensure that:

* the prepared OSM extract was generated successfully;
* mappings explicitly marked as `EXCLUDE` are not part of the primary feature
  scope;
* every `REFINE` mapping has a corresponding preparation rule;
* the resulting dataset remains a valid OSM PBF file with the references
  required for subsequent geometry reconstruction and import.

This validation does not redefine the semantic scope. It verifies that the
outputs produced by the preparation pipeline are structurally and
configuration-wise ready for the subsequent PostGIS import stage.


#### 5.1 Verify our  Files

In [32]:
# Verify that the preparation outputs exist.
assert PRIMARY_SCOPE_PATH.exists(), (
    f"Prepared PBF not found: {PRIMARY_SCOPE_PATH}"
)

assert PREPARATION_RULES_PATH.exists(), (
    f"Preparation rules not found: {PREPARATION_RULES_PATH}"
)

print("Preparation artifacts found.")
print(f"PBF:   {PRIMARY_SCOPE_PATH}")
print(f"Rules: {PREPARATION_RULES_PATH}")

Preparation artifacts found.
PBF:   ..\processed\01_primary_scope.osm.pbf
Rules: ..\filters\03_preparation_rules.json


#### 5.2 Verify PBF with Osmium

In [33]:
# Inspect the prepared PBF and verify that Osmium can read it.
result = subprocess.run(
    [
        "osmium",
        "fileinfo",
        "--extended",
        "--no-crc",
        str(PRIMARY_SCOPE_PATH),
    ],
    capture_output=True,
    text=True,
    encoding="utf-8",
    errors="replace",
    check=True,
)

print(result.stdout)

File:
  Name: ..\processed\01_primary_scope.osm.pbf
  Format: PBF
  Compression: none
  Size: 27066102
Header:
  Bounding boxes:
    (-83.23104,-4.25732,-66.8147199,16.5940699)
  With history: no
  Options:
    generator=osmium/1.19.0
    osmosis_replication_base_url=https://download.geofabrik.de/south-america/colombia-updates
    osmosis_replication_sequence_number=4900
    osmosis_replication_timestamp=2026-09-01T20:20:50Z
    pbf_dense_nodes=true
    pbf_optional_feature_0=Sort.Type_then_ID
    sorting=Type_then_ID
    timestamp=2026-09-01T20:20:50Z
Data:
  Bounding box: (-84.3166355,-52.8027348,-64.8146352,16.1306265)
  Timestamps:
    First: 2007-11-11T02:53:42Z
    Last: 2026-09-01T18:48:07Z
  Objects ordered (by type and id): yes
  Multiple versions of same object: no
  CRC32: not calculated (use --crc/-c to enable)
  Number of changesets: 0
  Number of nodes: 3826389
  Number of ways: 136379
  Number of relations: 3666
  Smallest changeset ID: 0
  Smallest node ID: 60175792
  S

#### 5.3 Verify that the EXCLUDE mapping has disappeared

In [34]:
# Verify that explicitly excluded primary mappings are absent
# from the prepared feature population.
excluded_scope = validated_scope[
    validated_scope["decision"] == "EXCLUDE"
].copy()

for _, row in excluded_scope.iterrows():
    expression = (
        f"{row['osm_key']}={row['osm_value']}"
    )

    result = subprocess.run(
        [
            "osmium",
            "tags-count",
            str(PRIMARY_SCOPE_PATH),
            expression,
        ],
        capture_output=True,
        text=True,
        encoding="utf-8",
        errors="replace",
        check=True,
    )

    print(f"Excluded mapping checked: {expression}")
    print(result.stdout.strip() or "No matching features found.")

Excluded mapping checked: amenity=parking_entrance
414	"amenity"	"parking_entrance"


#### Interpretation of result
The primary extraction excludes `amenity=parking_entrance` from the active
ColMaps feature scope.
A post-extraction inspection identified 414 objects carrying this tag in the
prepared PBF, compared with 1,044 occurrences in the original candidate
population.

These remaining objects do not indicate that the mapping is part of the
ColMaps feature scope. The primary extraction preserves referenced OSM
elements required by selected ways and relations, and therefore some objects
may remain in the PBF as structural dependencies.

Consequently, exclusion at this stage is interpreted semantically: objects
tagged only as `amenity=parking_entrance` are not eligible for construction as
independent ColMaps features during subsequent transformation and import.
Structural OSM elements are preserved to avoid compromising dataset integrity.


#### 5.4 Preparation Consistency Check

A final consistency check verifies that the semantic preparation specification
matches the validated feature scope.

Every mapping classified as `REFINE` must have exactly one corresponding
preparation rule, while mappings classified as `EXCLUDE` must not be part of
the active feature scope. This ensures that the preparation configuration can
be consumed deterministically by the subsequent transformation and import
stage.


**Check 1 - The 4 REFINE have rule**

In [35]:
# Build mapping identifiers from the validated scope.
refined_mappings = set(
    validated_scope.loc[
        validated_scope["decision"] == "REFINE",
        "osm_key",
    ]
    + "="
    + validated_scope.loc[
        validated_scope["decision"] == "REFINE",
        "osm_value",
    ]
)

specified_mappings = set(preparation_rules.keys())

assert refined_mappings == specified_mappings, (
    "Preparation rules do not match the validated "
    "REFINE mappings."
)

print(
    f"REFINE mappings validated: "
    f"{len(refined_mappings)}"
)

REFINE mappings validated: 4


**Check 2 - None `EXCLUDE` is in active_scope**

In [36]:
# Verify that excluded mappings are not part of the
# active semantic feature scope.
active_mappings = set(
    active_scope["osm_key"]
    + "="
    + active_scope["osm_value"]
)

excluded_mappings = set(
    validated_scope.loc[
        validated_scope["decision"] == "EXCLUDE",
        "osm_key",
    ]
    + "="
    + validated_scope.loc[
        validated_scope["decision"] == "EXCLUDE",
        "osm_value",
    ]
)

unexpected_active_exclusions = (
    active_mappings & excluded_mappings
)

assert not unexpected_active_exclusions, (
    "Excluded mappings found in active scope: "
    f"{unexpected_active_exclusions}"
)

print(
    f"Excluded mappings validated: "
    f"{len(excluded_mappings)}"
)

Excluded mappings validated: 1


**Check 3 - Complete Summary** 

In [37]:
print("Preparation consistency checks passed.")
print()
print(f"Validated mappings: {len(validated_scope)}")
print(f"Active mappings:    {len(active_scope)}")
print(f"RETAIN mappings:    {(validated_scope['decision'] == 'RETAIN').sum()}")
print(f"REFINE mappings:    {(validated_scope['decision'] == 'REFINE').sum()}")
print(f"EXCLUDE mappings:   {(validated_scope['decision'] == 'EXCLUDE').sum()}")
print(f"Preparation rules:  {len(preparation_rules)}")

Preparation consistency checks passed.

Validated mappings: 115
Active mappings:    114
RETAIN mappings:    110
REFINE mappings:    4
EXCLUDE mappings:   1
Preparation rules:  4


## 6. Final Outputs

The preparation pipeline has produced a reduced and structurally valid OSM
dataset together with the semantic rules required to construct the ColMaps
feature population.

The primary OSM extract preserves the referenced elements required for OSM
geometry reconstruction. Consequently, the presence of an object in the
prepared PBF does not necessarily imply that it represents an eligible ColMaps
feature. Final semantic eligibility is determined by the validated feature
scope and preparation rules.

The prepared dataset is now promoted to the canonical ColMaps OSM input:

`processed/colombia-colmaps.osm.pbf`

Together with `filters/02_validated_feature_scope.csv` and
`filters/03_preparation_rules.json`, this dataset provides the reproducible
input required by the subsequent PostGIS import and transformation stage.

### 6.1 Create the final PBF 
Since 01_primary_scope.osm.pbf is already the structurally correct file, we do not filter the original 311 MB file again. We simply rename it to the final filename.

In [38]:
import shutil


# Promote the validated primary extract to the canonical
# ColMaps prepared dataset.
shutil.copy2(
    PRIMARY_SCOPE_PATH,
    OUTPUT_PBF_PATH,
)

assert OUTPUT_PBF_PATH.exists()

print(f"Final prepared dataset: {OUTPUT_PBF_PATH}")

Final prepared dataset: ..\processed\colombia-colmaps.osm.pbf


### 6.2 Final reproducible summary

In [39]:
raw_size_mb = RAW_PBF_PATH.stat().st_size / (1024 ** 2)
final_size_mb = OUTPUT_PBF_PATH.stat().st_size / (1024 ** 2)

reduction_percent = (
    1 - OUTPUT_PBF_PATH.stat().st_size
    / RAW_PBF_PATH.stat().st_size
) * 100

print("ColMaps OSM dataset preparation complete.")
print()
print(f"Raw dataset:      {raw_size_mb:.2f} MiB")
print(f"Prepared dataset: {final_size_mb:.2f} MiB")
print(f"Size reduction:   {reduction_percent:.1f}%")
print()
print(f"Validated mappings: {len(validated_scope)}")
print(f"Active mappings:    {len(active_scope)}")
print(f"Preparation rules:  {len(preparation_rules)}")
print()
print(f"Output: {OUTPUT_PBF_PATH}")

ColMaps OSM dataset preparation complete.

Raw dataset:      313.28 MiB
Prepared dataset: 25.81 MiB
Size reduction:   91.8%

Validated mappings: 115
Active mappings:    114
Preparation rules:  4

Output: ..\processed\colombia-colmaps.osm.pbf


### 6.3 Final Observations

The ColMaps OSM preparation pipeline reduces the original Colombia extract by
approximately 91.8% while preserving the OSM elements required for structural
integrity and geometry reconstruction.

The resulting PBF contains the active primary OSM scope, while semantic
eligibility is governed by the validated feature-scope and preparation-rule
specifications. This separation prevents structural OSM dependencies from
being incorrectly interpreted as application-level features.

The prepared dataset and its associated specifications are ready for the
subsequent PostGIS import and transformation stage.
